In [1]:
from pathlib import Path

# Set this to your project folder if the kernel starts elsewhere.
PROJECT_ROOT = Path.cwd().resolve()
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT = PROJECT_ROOT / "data" / "processed" / "dengue_cleaned"
WEATHER_DIR = OUTPUT / "weather"
RUN_WEATHER_SAMPLE = False  # Enable only when you want a live API test.

for directory in [RAW_DATA_DIR, OUTPUT, WEATHER_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_ROOT)
print("Input folder:", RAW_DATA_DIR)
print("Output folder:", OUTPUT)


Project folder: C:\Users\User\Desktop\SAS_gc
Input folder: C:\Users\User\Desktop\SAS_gc\data\raw
Output folder: C:\Users\User\Desktop\SAS_gc\data\processed\dengue_cleaned


# Load and Clean dengue snapshots

In [3]:
import zipfile
from pathlib import Path
import re
import pandas as pd
import requests
import json
import time

# Location of the uploaded ZIP
zip_path = RAW_DATA_DIR / "sgcharts_dengue.zip"

# Create an extraction folder
extract_folder = OUTPUT / "sgcharts"
extract_folder.mkdir(exist_ok=True)

# Extract all CSVs
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_folder)

print("Extraction completed!")

csv_files = sorted(extract_folder.rglob("*.csv"))

print("Total CSV files:", len(csv_files))

for file in csv_files[:10]:
    print(file.name)

# Now we will need to remove the incorrect lat and long file
cluster_files = sorted(
    p for p in extract_folder.rglob("*.csv")
    if p.parent.name == "csv"
    and re.fullmatch(r"\d{6}-clusters\.csv", p.name)
)

print("Selected cluster files:", len(cluster_files))

for path in cluster_files[:5]:
    print(path.relative_to(extract_folder))


Extraction completed!
Total CSV files: 347
150703-clusters.csv
150716-clusters.csv
150806-clusters.csv
150811-clusters.csv
150814-clusters.csv
150817-clusters.csv
150821-clusters.csv
150825-clusters.csv
150828-clusters.csv
150831-clusters.csv
Selected cluster files: 256
sgcharts\csv\150703-clusters.csv
sgcharts\csv\150716-clusters.csv
sgcharts\csv\150806-clusters.csv
sgcharts\csv\150811-clusters.csv
sgcharts\csv\150814-clusters.csv


### 1.1 Inspect and combine source files

In [4]:
sample_indices = sorted({
    0,
    len(cluster_files) // 2,
    len(cluster_files) - 1,
})

for i in sample_indices:
    path = cluster_files[i]

    sample = pd.read_csv(
        path,
        header=None,
        dtype="string",
        encoding="utf-8-sig",
    )

    # print(f"\nFile: {path.name}")
    # print("Rows:", len(sample))
    # print("Columns:", sample.shape[1])
    # display(sample.head(3))


OUTPUT = PROJECT_ROOT / "data" / "processed" / "dengue_cleaned"
OUTPUT.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    "location_cases",
    "address",
    "latitude",
    "longitude",
    "cluster_number",
    "recent_cluster_cases",
    "total_cluster_cases",
    "snapshot_date_raw",
    "month_raw",
]

frames = []
file_records = []

assert cluster_files, "Run the cluster_files selection cell first."

#check if any column differs
for path in cluster_files:
    df = pd.read_csv(
        path,
        header=None,
        dtype="string",
        encoding="utf-8-sig",
        keep_default_na=False,
    )

    if df.shape[1] != 9:
        raise ValueError(
            f"{path.name}: expected 9 columns, found {df.shape[1]}"
        )

    df.columns = COLUMNS

    # Record where each observation came from just in case
    df["source_file"] = str(path.relative_to(extract_folder))
    df["source_row"] = range(1, len(df) + 1)
    df["filename_date_raw"] = path.name[:6]

    frames.append(df)
    file_records.append({
        "source_file": str(path.relative_to(extract_folder)),
        "filename_date_raw": path.name[:6],
        "raw_rows": len(df),
    })

#the name of the uncleaned table
raw = pd.concat(frames, ignore_index=True)
manifest = pd.DataFrame(file_records)

print("Files combined:", len(manifest))
print("Total rows:", len(raw))
display(raw.head(3))


Files combined: 256
Total rows: 56976


,location_cases,address,latitude,longitude,cluster_number,recent_cluster_cases,total_cluster_cases,snapshot_date_raw,month_raw,source_file,source_row,filename_date_raw
0,4,bishan street 22 (block 232),1.358286,103.845226,1,24,83,150703,7,sgcharts\csv\150703-clusters.csv,1,150703
1,7,bishan street 22 (block 233),1.358639,103.8452589,1,24,83,150703,7,sgcharts\csv\150703-clusters.csv,2,150703
2,3,bishan street 22 (block 234),1.35839,103.845955,1,24,83,150703,7,sgcharts\csv\150703-clusters.csv,3,150703


### 1.2 Clean dates, addresses and numeric values


In [5]:
#make a copy
clean = raw.copy()

#parse the observation date and filename date independently.
clean["snapshot_date"] = pd.to_datetime(
    clean["snapshot_date_raw"].str.strip(),
    format="%y%m%d",
    errors="coerce",
)

#file name date
clean["filename_date"] = pd.to_datetime(
    clean["filename_date_raw"],
    format="%y%m%d",
    errors="coerce",
)

#preserve the original address and create a cleaned version.
clean["address_clean"] = (
    clean["address"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

#numeric conversion
numeric_columns = [
    "latitude",
    "longitude",
    "location_cases",
    "recent_cluster_cases",
    "total_cluster_cases",
]

#preserve original numeric text before conversion.
for column in numeric_columns:
    clean[f"{column}_raw"] = raw[column]

    clean[column] = pd.to_numeric(
        raw[column].str.strip(),
        errors="coerce",
    )

display(
    clean[
        ["snapshot_date", "address_clean", "latitude", "longitude"]
    ].head()
)


,snapshot_date,address_clean,latitude,longitude
0,2015-07-03,bishan street 22 (block 232),1.358286,103.845226
1,2015-07-03,bishan street 22 (block 233),1.358639,103.845259
2,2015-07-03,bishan street 22 (block 234),1.35839,103.845955
3,2015-07-03,bishan street 22 (block 235),1.358719,103.846477
4,2015-07-03,bishan street 22 (block 236),1.359041,103.846849


### 1.3 Flag questionable records


In [6]:
clean["date_problem"] = (
    clean["snapshot_date"].isna()
    | clean["filename_date"].isna()
    | clean["snapshot_date"].ne(clean["filename_date"]) #this is to ensure consistencies within the file
)

#broad geographic screening as the lat and long must between sg boundaries
coordinates_ok = (
    clean["latitude"].between(1.1, 1.6)
    & clean["longitude"].between(103.5, 104.2)
).fillna(False)

clean["coordinate_problem"] = ~coordinates_ok

clean["address_problem"] = (
    clean["address_clean"].fillna("").eq("")
)

count_columns = [
    "location_cases",
    "recent_cluster_cases",
    "total_cluster_cases",
]

counts = clean[count_columns]

#identify missing, negative, or non-integer dengue case counts
clean["count_problem"] = (
    counts.isna().any(axis=1)
    | counts.lt(0).fillna(False).any(axis=1)
    | counts.mod(1).ne(0).fillna(False).any(axis=1)
)

FLAGS = [
    "date_problem",
    "coordinate_problem",
    "address_problem",
    "count_problem",
]

clean["needs_review"] = clean[FLAGS].any(axis=1)

print("Problems by category:")
print(clean[FLAGS].sum())

print("\nRows with any problem:", int(clean["needs_review"].sum()))
display(clean.loc[clean["needs_review"]].head(10))


Problems by category:
date_problem          0
coordinate_problem    0
address_problem       0
count_problem         0
dtype: Int64

Rows with any problem: 0


,location_cases,address,latitude,longitude,cluster_number,recent_cluster_cases,total_cluster_cases,snapshot_date_raw,month_raw,source_file,...,latitude_raw,longitude_raw,location_cases_raw,recent_cluster_cases_raw,total_cluster_cases_raw,date_problem,coordinate_problem,address_problem,count_problem,needs_review


### 1.4 Check duplicate records


In [7]:
duplicate_mask = raw.duplicated(
    subset=COLUMNS,
    keep="first",
)

# Save all members of duplicate groups for inspection.
duplicate_groups = clean.loc[
    raw.duplicated(subset=COLUMNS, keep=False)
].copy()

deduplicated = clean.loc[~duplicate_mask].copy()

deduplicated = deduplicated.sort_values(
    ["snapshot_date", "source_file", "source_row"]
).reset_index(drop=True)

# Keep flagged observations available for review.
rows_to_review = deduplicated.loc[
    deduplicated["needs_review"]
].copy()

# Dates and coordinates are essential for the next spatial step.
# Count/address flags remain visible but do not automatically
# disqualify a valid location from a presence-based analysis.
deduplicated["spatial_ready"] = ~(
    deduplicated["date_problem"]
    | deduplicated["coordinate_problem"]
)

print("Exact duplicate rows removed:", int(duplicate_mask.sum()))
print("Rows after deduplication:", len(deduplicated))
print("Rows requiring review:", len(rows_to_review))
print("Spatial-ready rows:", int(deduplicated["spatial_ready"].sum()))

display(duplicate_groups)


Exact duplicate rows removed: 3
Rows after deduplication: 56973
Rows requiring review: 0
Spatial-ready rows: 56973


,location_cases,address,latitude,longitude,cluster_number,recent_cluster_cases,total_cluster_cases,snapshot_date_raw,month_raw,source_file,...,latitude_raw,longitude_raw,location_cases_raw,recent_cluster_cases_raw,total_cluster_cases_raw,date_problem,coordinate_problem,address_problem,count_problem,needs_review
27876,1,ang mo kio avenue 10 (block 472),1.363185,103.858097,78,3,5,190726,7,sgcharts\csv\190726-clusters.csv,...,1.3631851,103.858097,1,3,5,False,False,False,False,False
27877,1,ang mo kio avenue 10 (block 472),1.363185,103.858097,78,3,5,190726,7,sgcharts\csv\190726-clusters.csv,...,1.3631851,103.858097,1,3,5,False,False,False,False,False
51958,1,sarkies road,1.314316,103.836788,24,15,111,200807,8,sgcharts\csv\200807-clusters.csv,...,1.314316,103.836788,1,15,111,False,False,False,False,False
51963,1,sarkies road,1.314316,103.836788,24,15,111,200807,8,sgcharts\csv\200807-clusters.csv,...,1.314316,103.836788,1,15,111,False,False,False,False,False
54800,1,sarkies road,1.314316,103.836788,15,4,135,200918,9,sgcharts\csv\200918-clusters.csv,...,1.314316,103.836788,1,4,135,False,False,False,False,False
54805,1,sarkies road,1.314316,103.836788,15,4,135,200918,9,sgcharts\csv\200918-clusters.csv,...,1.314316,103.836788,1,4,135,False,False,False,False,False


### 1.5 Inspect gaps between snapshots


In [9]:
manifest["snapshot_date"] = pd.to_datetime(
    manifest["filename_date_raw"],
    format="%y%m%d",
    errors="raise",
)

coverage = (
    manifest.groupby("snapshot_date", as_index=False)
    .agg(
        source_files=("source_file", "nunique"),
        raw_rows=("raw_rows", "sum"),
    )
)

row_summary = (
    deduplicated.groupby("filename_date", as_index=False)
    .agg(
        retained_rows=("source_file", "size"),
        spatial_ready_rows=("spatial_ready", "sum"),
        review_rows=("needs_review", "sum"),
    )
    .rename(columns={"filename_date": "snapshot_date"})
)

coverage = coverage.merge(
    row_summary,
    on="snapshot_date",
    how="left",
    validate="one_to_one",
).sort_values("snapshot_date")

coverage["gap_days"] = coverage["snapshot_date"].diff().dt.days

print("Snapshot dates:", len(coverage))
print("Date range:", coverage["snapshot_date"].min().date(),"to",coverage["snapshot_date"].max().date(),)

print("\nLargest gaps:")
display(coverage.nlargest(10, "gap_days"))

print("\nSmallest snapshots:")
display(coverage.nsmallest(10, "raw_rows"))


Snapshot dates: 256
Date range: 2015-07-03 to 2020-11-06

Largest gaps:


,snapshot_date,source_files,raw_rows,retained_rows,spatial_ready_rows,review_rows,gap_days
191,2019-04-03,1,101,101,101,0,64.0
255,2020-11-06,1,622,622,622,0,49.0
254,2020-09-18,1,2108,2107,2107,0,42.0
253,2020-08-07,1,3000,2999,2999,0,35.0
252,2020-07-03,1,1965,1965,1965,0,28.0
2,2015-08-06,1,182,182,182,0,21.0
52,2016-05-24,1,147,147,147,0,15.0
96,2017-04-03,1,6,6,6,0,14.0
132,2017-12-18,1,12,12,12,0,14.0
1,2015-07-16,1,160,160,160,0,13.0



Smallest snapshots:


,snapshot_date,source_files,raw_rows,retained_rows,spatial_ready_rows,review_rows,gap_days
95,2017-03-20,1,2,2,2,0,7.0
101,2017-05-08,1,2,2,2,0,6.0
142,2018-02-26,1,2,2,2,0,7.0
122,2017-10-02,1,5,5,5,0,7.0
143,2018-03-05,1,5,5,5,0,7.0
96,2017-04-03,1,6,6,6,0,14.0
100,2017-05-02,1,8,8,8,0,8.0
102,2017-05-15,1,9,9,9,0,7.0
98,2017-04-17,1,10,10,10,0,7.0
120,2017-09-18,1,10,10,10,0,7.0


### 1.6 Save cleaned data and audit reports


In [11]:
# Preserve an unmodified combined copy.
raw.to_csv(OUTPUT / "dengue_locations_raw.csv", index=False)

# Main cleaned table: retains quality flags and questionable rows.
deduplicated.to_csv(
    OUTPUT / "dengue_locations_clean.csv", index=False
)

rows_to_review.to_csv(
    OUTPUT / "rows_to_review.csv", index=False
)

duplicate_groups.to_csv(
    OUTPUT / "duplicate_groups.csv", index=False
)

manifest.to_csv(
    OUTPUT / "file_manifest.csv", index=False
)

coverage.to_csv(
    OUTPUT / "snapshot_coverage.csv", index=False
)



### 1.7 Load the working dengue table


In [12]:
import pandas as pd
from pathlib import Path

OUTPUT = PROJECT_ROOT / "data" / "processed" / "dengue_cleaned"

audit = pd.read_csv(
    OUTPUT / "dengue_locations_clean.csv",
    parse_dates=["snapshot_date"],
    dtype={"cluster_number": "string"},
)

# Check before removing the quality-control columns.
assert audit["needs_review"].eq(False).all(), \
    "Some rows still require review."

KEEP = [
    "snapshot_date",
    "address_clean",
    "latitude",
    "longitude",
    "cluster_number",
    "location_cases",
    "recent_cluster_cases",
    "total_cluster_cases",
    "source_file",
    "source_row",
]

dengue = (
    audit[KEEP]
    .rename(columns={"address_clean": "address"})
    .copy()
)

dengue.to_csv(
    OUTPUT / "dengue_locations.csv",
    index=False,
)

print("Working table:", dengue.shape)
display(dengue.head())


Working table: (56973, 10)


,snapshot_date,address,latitude,longitude,cluster_number,location_cases,recent_cluster_cases,total_cluster_cases,source_file,source_row
0,2015-07-03,bishan street 22 (block 232),1.358286,103.845226,1,4,24,83,sgcharts\csv\150703-clusters.csv,1
1,2015-07-03,bishan street 22 (block 233),1.358639,103.845259,1,7,24,83,sgcharts\csv\150703-clusters.csv,2
2,2015-07-03,bishan street 22 (block 234),1.358390,103.845955,1,3,24,83,sgcharts\csv\150703-clusters.csv,3
3,2015-07-03,bishan street 22 (block 235),1.358719,103.846477,1,1,24,83,sgcharts\csv\150703-clusters.csv,4
4,2015-07-03,bishan street 22 (block 236),1.359041,103.846849,1,4,24,83,sgcharts\csv\150703-clusters.csv,5


## 2. Assign locations to planning areas
- Basically checking where does the lat long falls into and aissgn a location

In [15]:
import geopandas as gpd
from bs4 import BeautifulSoup
# Replace with your actual uploaded filename
boundary_path = RAW_DATA_DIR / "Sg_boundaryArea.geojson"

# Read the GeoJSON file
areas = gpd.read_file(boundary_path)

#look at this, the polygons mainly contains the lat long of an named area
display(areas.head(2))

areas.columns = areas.columns.str.upper()

required = {"PLN_AREA_N", "PLN_AREA_C"}

if not required.issubset(areas.columns):
    assert "DESCRIPTION" in areas.columns, \
        "Area attributes not found. Send me the displayed columns."

    def parse_description(value):
        soup = BeautifulSoup(str(value), "html.parser")
        result = {}

        for row in soup.find_all("tr"):
            cells = row.find_all(["th", "td"])
            if len(cells) == 2:
                key = cells[0].get_text(" ", strip=True).upper()
                result[key] = cells[1].get_text(" ", strip=True)

        return result

    attributes = pd.DataFrame(
        areas["DESCRIPTION"].map(parse_description).tolist(),
        index=areas.index,
    )

    for column in required:
        if column not in areas.columns:
            assert column in attributes.columns, \
                f"{column} not found. Send me the boundary preview."
            areas[column] = attributes[column]


areas = areas.set_geometry("GEOMETRY")

assert areas.crs is not None, "Boundary CRS is missing."
areas = areas.to_crs("EPSG:4326") #lat long format

areas = areas.rename(columns={
    "PLN_AREA_C": "area_id",
    "PLN_AREA_N": "area_name",
})

areas = areas[
    ["area_id", "area_name", areas.geometry.name]
].copy()

assert areas["area_id"].notna().all()
assert areas.geometry.notna().all()
assert (~areas.geometry.is_empty).all(), \
    "Some boundary geometries need inspection."

#combine polygon pieces belonging to the same planning area.
areas = areas.dissolve(by="area_id", as_index=False)

print("Total planning areas:", len(areas))
display(areas[["area_id", "area_name"]].head())

#converting the points into lat and long
points = gpd.GeoDataFrame(
    dengue.copy(),
    geometry=gpd.points_from_xy(
        dengue["longitude"],
        dengue["latitude"],
    ),
    crs="EPSG:4326",
)

#creating multiple points based on the rows we have
points["record_id"] = range(len(points))

joined = gpd.sjoin(
    points,
    areas,
    how="left", #ensure we maintain the original rows even if didnt match any
    predicate="within",
)

#detect any point matching more than one polygon.
multiple = joined["record_id"].duplicated(keep=False)

print(
    "Records matching multiple areas:",
    joined.loc[multiple, "record_id"].nunique(),
)

assert not multiple.any(), \
    "Multiple area matches found. Inspect before continuing."

unmatched = joined.loc[joined["area_id"].isna()].copy()

print("Total records:", len(joined))
print("Matched records:", int(joined["area_id"].notna().sum()))
print("Unmatched records:", len(unmatched))

display(
    unmatched[
        ["snapshot_date", "address", "latitude", "longitude"]
    ].head(15)
)


,OBJECTID,PLN_AREA_N,PLN_AREA_C,CA_IND,REGION_N,REGION_C,INC_CRC,FMEL_UPD_D,SHAPE.AREA,SHAPE.LEN,geometry
0,1,BEDOK,BD,N,EAST REGION,ER,5F00E6FF084F3364,20191223152014,2.173397e+07,21864.234336,"POLYGON ((103.93208 1.30555, 103.93208 1.30555..."
1,2,BOON LAY,BL,N,WEST REGION,WR,C96AED188C00B2FC,20191223152014,8.282808e+06,18542.688792,"POLYGON ((103.72042 1.32824, 103.72066 1.32865..."


Total planning areas: 55


,area_id,area_name
0,AM,ANG MO KIO
1,BD,BEDOK
2,BK,BUKIT BATOK
3,BL,BOON LAY
4,BM,BUKIT MERAH


Records matching multiple areas: 0
Total records: 56973
Matched records: 56973
Unmatched records: 0


,snapshot_date,address,latitude,longitude


### 2.2 Save planning-area assignments


In [16]:
result = pd.DataFrame(
    joined.drop(columns=["geometry", "index_right"], errors="ignore")
)

result.to_csv(
    OUTPUT / "dengue_locations_with_areas.csv",
    index=False,
)

result.loc[result["area_id"].isna()].to_csv(
    OUTPUT / "unmatched_locations.csv",
    index=False,
)

display(result.head(10))


,snapshot_date,address,latitude,longitude,cluster_number,location_cases,recent_cluster_cases,total_cluster_cases,source_file,source_row,record_id,area_id,area_name
0,2015-07-03,bishan street 22 (block 232),1.358286,103.845226,1,4,24,83,sgcharts\csv\150703-clusters.csv,1,0,BS,BISHAN
1,2015-07-03,bishan street 22 (block 233),1.358639,103.845259,1,7,24,83,sgcharts\csv\150703-clusters.csv,2,1,BS,BISHAN
2,2015-07-03,bishan street 22 (block 234),1.358390,103.845955,1,3,24,83,sgcharts\csv\150703-clusters.csv,3,2,BS,BISHAN
3,2015-07-03,bishan street 22 (block 235),1.358719,103.846477,1,1,24,83,sgcharts\csv\150703-clusters.csv,4,3,BS,BISHAN
4,2015-07-03,bishan street 22 (block 236),1.359041,103.846849,1,4,24,83,sgcharts\csv\150703-clusters.csv,5,4,BS,BISHAN
5,2015-07-03,bishan street 22 (block 237),1.359335,103.847097,1,1,24,83,sgcharts\csv\150703-clusters.csv,6,5,BS,BISHAN
6,2015-07-03,bishan street 22 (block 293),1.356245,103.845879,1,2,24,83,sgcharts\csv\150703-clusters.csv,7,6,BS,BISHAN
7,2015-07-03,bishan street 23 (block 220),1.357568,103.848743,1,2,24,83,sgcharts\csv\150703-clusters.csv,8,7,BS,BISHAN
8,2015-07-03,bishan street 23 (block 222),1.358069,103.848079,1,1,24,83,sgcharts\csv\150703-clusters.csv,9,8,BS,BISHAN
9,2015-07-03,bishan street 23 (block 223),1.358557,103.848049,1,5,24,83,sgcharts\csv\150703-clusters.csv,10,9,BS,BISHAN


## 3. Create the weekly snapshot calendar
We need to make two changes:


*   Select the latest available snapshot from each week
*   Make Sunday as the date of the snapshot for Weekly data



In [17]:
locations = result.copy()
locations["snapshot_date"] = pd.to_datetime(
    locations["snapshot_date"]
)

# Use the saved coverage report to retain the full snapshot timeline.
snapshots = pd.read_csv(
    OUTPUT / "snapshot_coverage.csv",
    parse_dates=["snapshot_date"],
)

snapshots["cutoff_date"] = (
    snapshots["snapshot_date"]
    .dt.to_period("W-SUN")
    .dt.end_time
    .dt.normalize()
)

selected = (
    snapshots.sort_values("snapshot_date")
    .drop_duplicates("cutoff_date", keep="last")
    [["cutoff_date", "snapshot_date"]]
)

# Include missing calendar weeks explicitly.
calendar = pd.DataFrame({
    "cutoff_date": pd.date_range(
        snapshots["cutoff_date"].min(),
        snapshots["cutoff_date"].max(),
        freq="W-SUN",
    )
})

calendar = calendar.merge(
    selected,
    on="cutoff_date",
    how="left",
    validate="one_to_one",
)

calendar["snapshot_available"] = calendar["snapshot_date"].notna()

calendar["snapshot_age_days"] = (
    calendar["cutoff_date"] - calendar["snapshot_date"]
).dt.days.astype("Int64")

print("Calendar weeks:", len(calendar))
print("Weeks with snapshots:", int(calendar["snapshot_available"].sum()))
print("Missing weeks:", int((~calendar["snapshot_available"]).sum()))

display(calendar.head(10))


Calendar weeks: 280
Weeks with snapshots: 247
Missing weeks: 33


,cutoff_date,snapshot_date,snapshot_available,snapshot_age_days
0,2015-07-05,2015-07-03,True,2
1,2015-07-12,NaT,False,<NA>
2,2015-07-19,2015-07-16,True,3
3,2015-07-26,NaT,False,<NA>
4,2015-08-02,NaT,False,<NA>
5,2015-08-09,2015-08-06,True,3
6,2015-08-16,2015-08-14,True,2
7,2015-08-23,2015-08-21,True,2
8,2015-08-30,2015-08-28,True,2
9,2015-09-06,2015-09-04,True,2


### 3.2 Mark accepted snapshots


In [18]:
#initial working assumption, not a verified completeness finding.
ASSUME_AVAILABLE_SNAPSHOTS_COMPLETE = True

#dates for inspection
excluded_snapshot_dates = pd.to_datetime([
    # "2017-03-20",
])

calendar["snapshot_accepted"] = (
    calendar["snapshot_available"]
    & ASSUME_AVAILABLE_SNAPSHOTS_COMPLETE
    & ~calendar["snapshot_date"].isin(excluded_snapshot_dates)
)

calendar["coverage_basis"] = "unavailable"

calendar.loc[
    calendar["snapshot_available"], "coverage_basis"
] = "available_not_accepted"

calendar.loc[
    calendar["snapshot_accepted"], "coverage_basis"
] = "assumed_complete_pending_review"


display(
    calendar[
        [
            "cutoff_date",
            "snapshot_date",
            "snapshot_available",
            "snapshot_accepted",
            "coverage_basis"
        ]
    ].head(10)
)


,cutoff_date,snapshot_date,snapshot_available,snapshot_accepted,coverage_basis
0,2015-07-05,2015-07-03,True,True,assumed_complete_pending_review
1,2015-07-12,NaT,False,False,unavailable
2,2015-07-19,2015-07-16,True,True,assumed_complete_pending_review
3,2015-07-26,NaT,False,False,unavailable
4,2015-08-02,NaT,False,False,unavailable
5,2015-08-09,2015-08-06,True,True,assumed_complete_pending_review
6,2015-08-16,2015-08-14,True,True,assumed_complete_pending_review
7,2015-08-23,2015-08-21,True,True,assumed_complete_pending_review
8,2015-08-30,2015-08-28,True,True,assumed_complete_pending_review
9,2015-09-06,2015-09-04,True,True,assumed_complete_pending_review


### 3.3 Count distinct reported locations


In [19]:
#inspecting unique cluster addresses combo
accepted_dates = calendar.loc[
    calendar["snapshot_accepted"], "snapshot_date"
]

selected_locations = locations.loc[
    locations["snapshot_date"].isin(accepted_dates)
].copy()

distinct_locations = selected_locations.drop_duplicates(
    subset=[
        "snapshot_date",
        "area_id",
        "address",
        "latitude",
        "longitude",
    ]
)

location_counts = (
    distinct_locations
    .groupby(["snapshot_date", "area_id"])
    .size()
    .rename("reported_location_count")
    .reset_index()
)

display(location_counts.head(10))


,snapshot_date,area_id,reported_location_count
0,2015-07-03,BD,9
1,2015-07-03,BM,3
2,2015-07-03,BS,49
3,2015-07-03,BT,4
4,2015-07-03,CK,3
5,2015-07-03,GL,6
6,2015-07-03,HG,2
7,2015-07-03,JE,2
8,2015-07-03,JW,3
9,2015-07-03,PG,4


### 3.4 Build the area–week panel and next-week target


In [20]:
area_list = areas[["area_id", "area_name"]].drop_duplicates()

assert area_list["area_id"].is_unique

panel = area_list.merge(calendar, how="cross")

panel = panel.merge(
    location_counts,
    on=["snapshot_date", "area_id"],
    how="left",
    validate="many_to_one",
)

print("Area–week rows:", len(panel))

accepted = panel["snapshot_accepted"]

panel.loc[accepted, "reported_location_count"] = (
    panel.loc[accepted, "reported_location_count"].fillna(0)
)

panel.loc[~accepted, "reported_location_count"] = pd.NA

panel["reported_location_count"] = (
    panel["reported_location_count"].astype("Int64")
)

panel["current_presence"] = (
    panel["reported_location_count"].gt(0).astype("Int64")
)

panel = panel.sort_values(
    ["area_id", "cutoff_date"]
).reset_index(drop=True)

future = panel[
    ["area_id", "cutoff_date", "current_presence", "snapshot_date"]
].copy()

future = future.rename(columns={
    "current_presence": "presence_next_week",
    "snapshot_date": "target_snapshot_date",
})

# Align next week's outcome with this week's prediction cutoff.
future["cutoff_date"] -= pd.Timedelta(days=7)

panel = panel.merge(
    future,
    on=["area_id", "cutoff_date"],
    how="left",
    validate="one_to_one",
)

panel["presence_next_week"] = (
    panel["presence_next_week"].astype("Int64")
)

panel["target_observed"] = panel["presence_next_week"].notna()


Area–week rows: 15400


### 3.5 Validate and save next-week targets


In [ ]:
assert not panel.duplicated(
    ["area_id", "cutoff_date"]
).any()

assert len(panel) == len(area_list) * len(calendar)

assert panel.loc[
    ~panel["snapshot_accepted"], "current_presence"
].isna().all()

assert panel["presence_next_week"].dropna().isin([0, 1]).all()

known_targets = panel["presence_next_week"].dropna()

print("Total area–week rows:", len(panel))
print("Rows with observed targets:", len(known_targets))
print("Rows with unknown targets:", panel["presence_next_week"].isna().sum())

if len(known_targets):
    print("Positive-target proportion:", f"{known_targets.mean():.1%}")

display(
    panel.loc[
        panel["area_id"].eq("AM"),
        [
            "area_name",
            "cutoff_date",
            "snapshot_date",
            "reported_location_count",
            "current_presence",
            "presence_next_week",
        ],
    ].head(12)
)

panel.to_csv(
    OUTPUT / "sg_area_week_dengue.csv",
    index=False,
)

calendar.to_csv(
    OUTPUT / "weekly_snapshot_calendar.csv",
    index=False,
)


Total area–week rows: 15400
Rows with observed targets: 13530
Rows with unknown targets: 1870
Positive-target proportion: 27.5%


,area_name,cutoff_date,snapshot_date,reported_location_count,current_presence,presence_next_week
0,ANG MO KIO,2015-07-05,2015-07-03,0,0,<NA>
1,ANG MO KIO,2015-07-12,NaT,<NA>,<NA>,1
2,ANG MO KIO,2015-07-19,2015-07-16,2,1,<NA>
3,ANG MO KIO,2015-07-26,NaT,<NA>,<NA>,<NA>
4,ANG MO KIO,2015-08-02,NaT,<NA>,<NA>,1
5,ANG MO KIO,2015-08-09,2015-08-06,3,1,0
6,ANG MO KIO,2015-08-16,2015-08-14,0,0,0
7,ANG MO KIO,2015-08-23,2015-08-21,0,0,1
8,ANG MO KIO,2015-08-30,2015-08-28,3,1,1
9,ANG MO KIO,2015-09-06,2015-09-04,1,1,1


## 4. Prepare Open-Meteo weather features


*   Select a representative weather point for each locations



In [23]:
import pandas as pd
import geopandas as gpd

# Work in Singapore's projected coordinate system.
areas_projected = areas.to_crs("EPSG:3414")

# Choose a point inside each planning area.
representative_points = (
    areas_projected.geometry
    .representative_point()
    .to_crs("EPSG:4326")
)

weather_locations = areas[
    ["area_id", "area_name"]
].copy()

weather_locations["latitude"] = representative_points.y
weather_locations["longitude"] = representative_points.x

weather_locations = weather_locations.reset_index(drop=True)

assert weather_locations["area_id"].is_unique
assert len(weather_locations) == 55

weather_locations.to_csv(
    OUTPUT / "weather_locations.csv",
    index=False,
)

display(weather_locations.head())


,area_id,area_name,latitude,longitude
0,AM,ANG MO KIO,1.376729,103.842565
1,BD,BEDOK,1.325001,103.930869
2,BK,BUKIT BATOK,1.356030,103.752601
3,BL,BOON LAY,1.315360,103.707792
4,BM,BUKIT MERAH,1.274457,103.821630


### 4.2 Optional Ang Mo Kio API test


In [24]:
if RUN_WEATHER_SAMPLE:
    panel = pd.read_csv(
        OUTPUT / "sg_area_week_dengue.csv",
        parse_dates=["cutoff_date"],
    )
    
    start_date = (
        panel["cutoff_date"].min() - pd.Timedelta(days=28)
    ).strftime("%Y-%m-%d")
    
    end_date = panel["cutoff_date"].max().strftime("%Y-%m-%d")
    
    #getting the end to end date
    print("Weather period:", start_date, "to", end_date)
    
    #Open-Meteo’s API
    WEATHER_DIR = OUTPUT / "weather"
    WEATHER_DIR.mkdir(parents=True, exist_ok=True)
    
    #taking one location to check
    location = weather_locations.loc[
        weather_locations["area_id"].eq("AM")
    ].iloc[0]
    
    params = {
        "latitude": float(location["latitude"]),
        "longitude": float(location["longitude"]),
        "start_date": start_date,
        "end_date": end_date,
        "daily": ",".join([
            "temperature_2m_mean",
            "temperature_2m_max",
            "temperature_2m_min",
            "precipitation_sum",
            "relative_humidity_2m_mean",
            "wind_speed_10m_mean",
            "soil_moisture_0_to_7cm_mean",
        ]),
        "models": "era5",
        "timezone": "Asia/Singapore",
        "temperature_unit": "celsius",
        "precipitation_unit": "mm",
        "wind_speed_unit": "kmh",
    }
    
    response = requests.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=120,
    )
    
    if not response.ok:
        raise RuntimeError(
            f"Weather request failed: {response.status_code}\n"
            f"{response.text[:1000]}"
        )
    
    data = response.json()
    
    #save the response and request settings for reproducibility.
    (WEATHER_DIR / "AM_response.json").write_text(
        json.dumps(
            {"request": params, "response": data},
            indent=2,
        ),
        encoding="utf-8",
    )
    
    weather_sample = pd.DataFrame(data["daily"]).rename(columns={
        "time": "date",
        "temperature_2m_mean": "temperature_mean_c",
        "temperature_2m_max": "temperature_max_c",
        "temperature_2m_min": "temperature_min_c",
        "precipitation_sum": "precipitation_mm",
        "relative_humidity_2m_mean": "relative_humidity_mean_pct",
        "wind_speed_10m_mean": "wind_speed_mean_kmh",
        "soil_moisture_0_to_7cm_mean": "soil_moisture_0_7cm_m3m3",
    })
    
    weather_sample["date"] = pd.to_datetime(weather_sample["date"])
    weather_sample.insert(0, "area_id", location["area_id"])
    
    
    weather_sample.to_csv(
        WEATHER_DIR / "AM_daily_weather.csv",
        index=False,
    )
    
    print("Area:", location["area_name"])
    print("Daily rows:", len(weather_sample))
    print("Units:", data["daily_units"])
    print("Returned coordinates:", data["latitude"], data["longitude"])
    
    display(weather_sample.head())
    print("\nMissing values:")
    print(weather_sample.isna().sum())
    
    # files.download(
    #     str(WEATHER_DIR / "AM_daily_weather.csv")
    # )
else:
    print("Live API sample skipped. Continue with the saved-weather workflow.")


Live API sample skipped. Continue with the saved-weather workflow.


### 4.3 Configure the full weather timeline


In [25]:
WEATHER_DIR = OUTPUT / "weather"
WEATHER_DIR.mkdir(parents=True, exist_ok=True)

weather_locations = pd.read_csv(
    OUTPUT / "weather_locations.csv",
    dtype={"area_id": "string"},
)

panel = pd.read_csv(
    OUTPUT / "sg_area_week_dengue.csv",
    parse_dates=["cutoff_date"],
)

start_date = (
    panel["cutoff_date"].min() - pd.Timedelta(days=28)
).strftime("%Y-%m-%d")

end_date = panel["cutoff_date"].max().strftime("%Y-%m-%d")

RENAME = {
    "temperature_2m_mean": "temperature_mean_c",
    "temperature_2m_max": "temperature_max_c",
    "temperature_2m_min": "temperature_min_c",
    "precipitation_sum": "precipitation_mm",
    "relative_humidity_2m_mean": "relative_humidity_mean_pct",
    "wind_speed_10m_mean": "wind_speed_mean_kmh",
    "soil_moisture_0_to_7cm_mean": "soil_moisture_0_7cm_m3m3",
}

expected_dates = pd.date_range(start_date, end_date, freq="D")

assert len(weather_locations) == 55
assert weather_locations["area_id"].is_unique


### Download reference


### 4.4 Original bulk downloader (disabled)


In [26]:
# weather_frames = []
# metadata_rows = []

# with requests.Session() as session:

#     for number, location in enumerate(
#         weather_locations.itertuples(index=False), start=1
#     ):
#         area_id = location.area_id

#         params = {
#             "latitude": float(location.latitude),
#             "longitude": float(location.longitude),
#             "start_date": start_date,
#             "end_date": end_date,
#             "daily": ",".join(RENAME),
#             "models": "era5",
#             "timezone": "Asia/Singapore",
#             "temperature_unit": "celsius",
#             "precipitation_unit": "mm",
#             "wind_speed_unit": "kmh",
#         }

#         cache_path = WEATHER_DIR / f"{area_id}_response.json"
#         data = None

#         if cache_path.exists():
#             cached = json.loads(cache_path.read_text(encoding="utf-8"))

#             if cached.get("request") == params:
#                 data = cached["response"]

#         if data is None:
#             response = session.get(
#                 "https://archive-api.open-meteo.com/v1/archive",
#                 params=params,
#                 timeout=120,
#             )

#             if not response.ok:
#                 raise RuntimeError(
#                     f"{area_id}: HTTP {response.status_code}\n"
#                     f"{response.text[:1000]}\n"
#                     f"Retry-After: {response.headers.get('Retry-After')}\n"
#                     "Completed downloads are saved. Rerun later."
#                 )

#             data = response.json()

#             cache_path.write_text(
#                 json.dumps({"request": params, "response": data}),
#                 encoding="utf-8",
#             )

#             time.sleep(15)

#         daily = pd.DataFrame(data["daily"])

#         missing_columns = set(RENAME) - set(daily.columns)
#         assert not missing_columns, (
#             f"{area_id}: missing variables {missing_columns}"
#         )

#         daily = daily.rename(columns={"time": "date", **RENAME})
#         daily["date"] = pd.to_datetime(daily["date"])
#         daily = daily.sort_values("date").reset_index(drop=True)

#         assert not daily["date"].duplicated().any(), area_id
#         assert pd.DatetimeIndex(daily["date"]).equals(
#             expected_dates
#         ), f"{area_id}: unexpected daily timeline"

#         daily.insert(0, "area_id", area_id)

#         daily.to_csv(
#             WEATHER_DIR / f"{area_id}_daily_weather.csv",
#             index=False,
#         )

#         weather_frames.append(daily)

#         metadata_rows.append({
#             "area_id": area_id,
#             "area_name": location.area_name,
#             "requested_latitude": location.latitude,
#             "requested_longitude": location.longitude,
#             "returned_latitude": data["latitude"],
#             "returned_longitude": data["longitude"],
#             "returned_elevation": data.get("elevation"),
#             "model": "era5",
#             "timezone": data["timezone"],
#         })

#         print(f"{number}/55: {location.area_name} — {len(daily)} days")

# weather_daily = pd.concat(weather_frames, ignore_index=True)
# weather_metadata = pd.DataFrame(metadata_rows)


### 4.5 Download checks

In [27]:
# assert weather_daily["area_id"].nunique() == 55
# assert len(weather_daily) == 55 * len(expected_dates)
# assert not weather_daily.duplicated(["area_id", "date"]).any()

# print("Daily weather rows:", len(weather_daily))

# print("\nMissing measurements:")
# display(
#     weather_daily[list(RENAME.values())]
#     .isna().sum().to_frame("missing")
# )

# weather_daily.to_csv(
#     OUTPUT / "sg_area_daily_weather.csv", index=False
# )

# weather_metadata.to_csv(
#     OUTPUT / "weather_source_metadata.csv", index=False
# )

# zip_path = OUTPUT / "sg_weather_data.zip"

# with zipfile.ZipFile(zip_path, "w") as z:
#     z.write(
#         OUTPUT / "sg_area_daily_weather.csv",
#         arcname="sg_area_daily_weather.csv"
#     )
#     z.write(
#         OUTPUT / "weather_source_metadata.csv",
#         arcname="weather_source_metadata.csv"
#     )

# #files.download(str(zip_path))

# grid_summary = (
#     weather_metadata
#     .groupby(["returned_latitude", "returned_longitude"])
#     .agg(
#         number_of_areas=("area_id", "size"),
#         area_names=("area_name", lambda x: ", ".join(x)),
#     )
#     .reset_index()
# )

# print("Distinct returned grid locations:", len(grid_summary))
# display(grid_summary)


### 4.6 Load previously downloaded weather


In [29]:
WEATHER_DIR = OUTPUT / "weather"

# Load existing files
for filename in ["sg_area_daily_weather.csv", "weather_source_metadata.csv"]:
    assert (WEATHER_DIR / filename).exists(), (
        f"Missing saved weather file: {WEATHER_DIR / filename}. "
        "Copy it from your existing weather checkpoint before continuing."
    )
weather_daily = pd.read_csv(
    WEATHER_DIR / "sg_area_daily_weather.csv",
    dtype={"area_id": "string"},
    parse_dates=["date"]
)

weather_metadata = pd.read_csv(
    WEATHER_DIR / "weather_source_metadata.csv",
    dtype={"area_id": "string"}
)

# Verify
print("Weather daily:", weather_daily.shape)
print("Weather metadata:", weather_metadata.shape)

display(weather_daily.head())
display(weather_metadata.head())


Weather daily: (109010, 9)
Weather metadata: (55, 9)


,area_id,date,temperature_mean_c,temperature_max_c,temperature_min_c,precipitation_mm,relative_humidity_mean_pct,wind_speed_mean_kmh,soil_moisture_0_7cm_m3m3
0,AM,2015-06-07,27.3,29.9,25.2,8.6,87,6.5,0.421
1,AM,2015-06-08,27.4,30.8,25.1,5.4,88,6.3,0.418
2,AM,2015-06-09,27.5,30.5,25.0,5.3,87,6.3,0.414
3,AM,2015-06-10,27.6,30.4,25.3,5.3,87,5.7,0.416
4,AM,2015-06-11,27.2,29.5,25.3,15.3,87,6.8,0.421


,area_id,area_name,requested_latitude,requested_longitude,returned_latitude,returned_longitude,returned_elevation,model,timezone
0,AM,ANG MO KIO,1.376729,103.842565,1.5,103.75,13.0,era5,Asia/Singapore
1,BD,BEDOK,1.325001,103.930869,1.5,104.00,15.0,era5,Asia/Singapore
2,BK,BUKIT BATOK,1.356030,103.752601,1.5,103.75,27.0,era5,Asia/Singapore
3,BL,BOON LAY,1.315360,103.707792,1.5,103.75,13.0,era5,Asia/Singapore
4,BM,BUKIT MERAH,1.274457,103.821630,1.5,103.75,27.0,era5,Asia/Singapore


### 4.7 Calculate historical weather features


In [30]:
weather_daily["date"] = pd.to_datetime(weather_daily["date"])

# Columns to average over each window.
MEAN_FEATURES = {
    "temperature_mean_c": "temp",
    "temperature_max_c": "daily_tmax",
    "temperature_min_c": "daily_tmin",
    "relative_humidity_mean_pct": "humidity",
    "wind_speed_mean_kmh": "wind",
    "soil_moisture_0_7cm_m3m3": "soil",
    "temperature_range_c": "temp_range",
}

feature_frames = []

for area_id, group in weather_daily.groupby("area_id"):
    daily = (
        group.sort_values("date")
        .set_index("date")
        .reindex(expected_dates)
    )
    daily.index.name = "cutoff_date"

    daily["temperature_range_c"] = (
        daily["temperature_max_c"] - daily["temperature_min_c"]
    )

    features = pd.DataFrame(index=daily.index)

    #precipitation totals.
    rain = daily["precipitation_mm"]

    #taking data weeks before as weeks before might affect weeks after
    features["precip_7d"] = rain.rolling(
        7, min_periods=7
    ).sum()

    features["precip_days8_28"] = rain.shift(7).rolling(
        21, min_periods=21
    ).sum()

    # Completeness indicators.
    features["precip_valid_days_7d"] = (
        rain.notna().astype(int).rolling(7, min_periods=7).sum()
    )

    features["precip_valid_days8_28"] = (
        rain.shift(7).notna().astype(int)
        .rolling(21, min_periods=21).sum()
    )

    # Temperature, humidity, wind and soil summaries.
    for column, prefix in MEAN_FEATURES.items():
        for window in [7, 28]:
            features[f"{prefix}_{window}d"] = (
                daily[column]
                .rolling(window, min_periods=window)
                .mean()
            )

            features[f"{prefix}_valid_days_{window}d"] = (
                daily[column].notna().astype(int)
                .rolling(window, min_periods=window)
                .sum()
            )

    features["area_id"] = area_id
    feature_frames.append(features.reset_index())

weather_features_daily = pd.concat(
    feature_frames, ignore_index=True
)

print("Daily features calculated.")


Daily features calculated.


### 4.8 Join weather to weekly dengue labels


In [31]:
panel = pd.read_csv(
    OUTPUT / "sg_area_week_dengue.csv",
    parse_dates=[
        "cutoff_date",
        "snapshot_date",
        "target_snapshot_date",
    ],
)

weekly_weather = weather_features_daily.loc[
    weather_features_daily["cutoff_date"].isin(panel["cutoff_date"])
].copy()

assert not weekly_weather.duplicated(
    ["area_id", "cutoff_date"]
).any()

model_panel = panel.merge(
    weekly_weather,
    on=["area_id", "cutoff_date"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

assert model_panel["_merge"].eq("both").all(), \
    "Some area–week rows have no weather match."

model_panel = model_panel.drop(columns="_merge")

assert len(model_panel) == len(panel)
assert not model_panel.duplicated(
    ["area_id", "cutoff_date"]
).any()

print("Combined area–week rows:", len(model_panel))


Combined area–week rows: 15400


### 4.9 Save modelling tables


In [ ]:
weekly_weather.to_csv(
    OUTPUT / "sg_area_week_weather.csv",
    index=False,
)

model_panel.to_csv(
    OUTPUT / "sg_dengue_weather_panel.csv",
    index=False,
)


## 5. Train and evaluate presence models


### 5.1 Define model features and observed targets


In [33]:
from pathlib import Path
import numpy as np
import pandas as pd


OUTPUT = PROJECT_ROOT / "data" / "processed" / "dengue_cleaned"
MODEL_DIR = OUTPUT / "modelling"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    OUTPUT / "sg_dengue_weather_panel.csv",
    dtype={"area_id": "string"},
    parse_dates=[
        "cutoff_date",
        "snapshot_date",
        "target_snapshot_date",
    ],
)

#binary target, 0 or 1, whether if there will be cluster the next week
TARGET = "presence_next_week"

#we are also using the 4 weeks ahead weather condition as weeks before might affect weeks after
WEATHER_FEATURES = [
    "precip_7d",
    "precip_days8_28",
    "temp_7d", "temp_28d",
    "daily_tmax_7d", "daily_tmax_28d",
    "daily_tmin_7d", "daily_tmin_28d",
    "humidity_7d", "humidity_28d",
    "wind_7d", "wind_28d",
    "soil_7d", "soil_28d",
    "temp_range_7d", "temp_range_28d",
]

required = [
    "area_id", "cutoff_date", "target_snapshot_date", TARGET,
    *WEATHER_FEATURES,
]
assert set(required).issubset(df.columns)
assert not df.duplicated(["area_id", "cutoff_date"]).any()
assert df[WEATHER_FEATURES].notna().all().all()

print("Master panel:", df.shape)
print("Observed targets:", df[TARGET].notna().sum())
print("Unknown targets:", df[TARGET].isna().sum())

#smooth annual cycle; avoids treating December and January as far apart
days_in_year = np.where(df["cutoff_date"].dt.is_leap_year, 366, 365)
annual_phase = (
    2 * np.pi
    * (df["cutoff_date"].dt.dayofyear - 1)
    / days_in_year
)

df["season_sin"] = np.sin(annual_phase)
df["season_cos"] = np.cos(annual_phase)

#only pick not NA target
model_df = df.loc[df[TARGET].notna()].copy()
model_df[TARGET] = model_df[TARGET].astype(int)

assert model_df[TARGET].isin([0, 1]).all()
assert model_df["target_snapshot_date"].notna().all()
assert (
    model_df["target_snapshot_date"] > model_df["cutoff_date"]
).all()

CONTEXT_FEATURES = ["area_id", "season_sin", "season_cos"]
FULL_FEATURES = CONTEXT_FEATURES + WEATHER_FEATURES

print("Modelling rows:", len(model_df))
print("Positive proportion:", model_df[TARGET].mean().round(3))


Master panel: (15400, 45)
Observed targets: 13530
Unknown targets: 1870
Modelling rows: 13530
Positive proportion: 0.275


C:\Users\User\AppData\Local\Temp\ipykernel_17980\1600153142.py:52: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  * (df["cutoff_date"].dt.dayofyear - 1)


### 5.2 Split training, validation and test periods


In [34]:
#validation data
VAL_START = pd.Timestamp("2019-01-01")
#test data
TEST_START = pd.Timestamp("2020-01-01")

#train anything before 2019
train = model_df.loc[
    (model_df["cutoff_date"] < VAL_START)
    & (model_df["target_snapshot_date"] < VAL_START)
].copy()

valid = model_df.loc[
    (model_df["cutoff_date"] >= VAL_START)
    & (model_df["cutoff_date"] < TEST_START)
    & (model_df["target_snapshot_date"] < TEST_START)
].copy()

test = model_df.loc[
    model_df["cutoff_date"] >= TEST_START
].copy()

for name, part in [
    ("train", train), ("validation", valid), ("test", test)
]:
    assert len(part) > 0, f"{name} is empty"
    assert part[TARGET].nunique() == 2, f"{name} needs both classes"

    print(
        name,
        "| rows:", len(part),
        "| weeks:", part["cutoff_date"].nunique(),
        "| positive rate:", round(part[TARGET].mean(), 3),
        "| dates:", part["cutoff_date"].min().date(),
        "to", part["cutoff_date"].max().date(),
    )

train.to_csv(MODEL_DIR / "train.csv", index=False)
valid.to_csv(MODEL_DIR / "validation.csv", index=False)
test.to_csv(MODEL_DIR / "test.csv", index=False)


train | rows: 9735 | weeks: 177 | positive rate: 0.21 | dates: 2015-07-12 to 2018-12-30
validation | rows: 2310 | weeks: 42 | positive rate: 0.392 | dates: 2019-01-06 to 2019-12-22
test | rows: 1430 | weeks: 26 | positive rate: 0.524 | dates: 2020-01-05 to 2020-11-01


In [40]:
#print(train.head())

### 5.3 Fit area/season and weather/context models


In [42]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

def make_pipeline(features, estimator):
    categorical = [c for c in features if c == "area_id"]
    numeric = [c for c in features if c != "area_id"]

    preprocessing = ColumnTransformer([
        ("numeric", StandardScaler(), numeric),
        ("area", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])

    return Pipeline([
        ("preprocessing", preprocessing),
        ("model", estimator),
    ])

experiments = {
    #only using area number and time
    "area_season": (
        CONTEXT_FEATURES,
        LogisticRegression(max_iter=4000),
    ),
    #using both weather and area and time
    "weather_logistic": (
        FULL_FEATURES,
        LogisticRegression(max_iter=4000),
    ),
    #ranodm forest test
    "weather_forest": (
        FULL_FEATURES,
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1,
        ),
    ),    
    
    "weather_only_logistic": (
        WEATHER_FEATURES,
        LogisticRegression(max_iter=2000),
    ),
    "weather_only_forest": (
        WEATHER_FEATURES,
        RandomForestClassifier(
            n_estimators=200,
            max_depth=10,
            min_samples_leaf=20,
            random_state=42,
            n_jobs=-1,
        ),
    ),
}

fitted = {}
validation_predictions = {
    "constant": np.full(len(valid), train[TARGET].mean())
}

for name, (features, estimator) in experiments.items():
    pipeline = make_pipeline(features, estimator)
    pipeline.fit(train[features], train[TARGET])

    fitted[name] = pipeline
    validation_predictions[name] = pipeline.predict_proba(
        valid[features]
    )[:, 1]

    print("Finished:", name)


Finished: area_season
Finished: weather_logistic
Finished: weather_forest
Finished: weather_only_logistic
Finished: weather_only_forest


### 5.4 Compare validation probabilities


In [43]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

def probability_metrics(y, probability):
    return {
        "average_precision": average_precision_score(y, probability),
        "roc_auc": roc_auc_score(y, probability),
        "brier_score": brier_score_loss(y, probability),
    }

validation_results = pd.DataFrame([
    {
        "model": name,
        **probability_metrics(valid[TARGET], probability),
    }
    for name, probability in validation_predictions.items()
]).sort_values("average_precision", ascending=False)

display(validation_results)
validation_results.to_csv(
    MODEL_DIR / "validation_results.csv", index=False
)


,model,average_precision,roc_auc,brier_score
1,area_season,0.867339,0.926401,0.164934
2,weather_logistic,0.862054,0.914498,0.169564
3,weather_forest,0.753396,0.786787,0.234090
4,weather_only_logistic,0.436671,0.562633,0.267747
5,weather_only_forest,0.408134,0.537382,0.273012
0,constant,0.392208,0.500000,0.271706
